# Truthprint W4 — Real baseline head-to-head on real MT

**Goal (review weakness W4):** run a *real* watermark detector for a token-level
baseline (self-contained KGW, Kirchenbauer et al. 2023) — and optionally SynthID/SIR
via MarkLLM — on the **same** watermark → real-NLLB-translation pipeline used for
Truthprint, then score an apples-to-apples comparison at 1% FPR.

**No manual annotation needed.** Run all cells. The last cell writes
`w4_results.zip` (contains `05_baseline_outputs.jsonl`, the scored comparison
table, and Truthprint's provenance numbers) — send it back.

Recommended: Kaggle/Colab with **GPU T4**, Internet **On**. CPU works but is slow.


## Cell 1 — environment detect + clone + install


In [ ]:
import os, sys, subprocess
BASE = '/kaggle/working' if os.path.isdir('/kaggle/working') else ('/content' if os.path.isdir('/content') else os.getcwd())
REPO = os.path.join(BASE, 'truthprint')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','https://github.com/leemgs/truthprint', REPO], check=True)
sys.path.insert(0, os.path.join(REPO, 'code'))
WORK = os.path.join(BASE, 'w4_work'); os.makedirs(WORK, exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers','sentencepiece','torch','--upgrade'], check=False)
import torch; print('cuda:', torch.cuda.is_available())


## Cell 2 — shared source facts + natural-language prompts
Reuse the closed-domain facts (same corpus family as the B2 multilingual run) so
every method watermarks the *same* content. `N_DOCS` controls scale.


In [ ]:
import json, random
from truthprint import challenge as ch
N_DOCS, SENTS = 50, 16          # more docs -> tighter CIs
rng = random.Random(20)
src_path = os.path.join(WORK, '01_source_items.jsonl')
facts_all = []
with open(src_path, 'w', encoding='utf-8') as f:
    for d in range(N_DOCS):
        facts = [ch.sample_fact(rng) for _ in range(SENTS)]
        rec = {'doc_id': f'D{d:04d}', 'facts': [dict(ch.ext_invariants(x), sent_id=f'D{d:04d}-s{i+1}') for i,x in enumerate(facts)]}
        f.write(json.dumps(rec, ensure_ascii=False)+'\n')
        facts_all.append((rec['doc_id'], facts))
# One English sentence per fact = the content each watermark method must carry.
def sentence(fact):
    return ch.realize(fact, 0, 1)
docs_text = {did: ' '.join(sentence(x) for x in facts) for did, facts in facts_all}
prompts = {did: f'Write a short factual incident report.\n' for did in docs_text}
print('docs:', len(docs_text)); print('example:', list(docs_text.values())[0][:160])


## Cell 3 — self-contained KGW watermark (real LLM generation + detection)
A faithful Kirchenbauer-style green-list watermark on a real HF model. This is the
guaranteed real token-level baseline; He et al. (2024) showed such signals do not
survive translation — we measure exactly that.


In [ ]:
import torch, math
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList
MODEL_ID = 'gpt2'      # swap for a larger/instruct model for more fluent text
GAMMA, DELTA, WM_KEY = 0.25, 2.0, 15485863
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device).eval()
V = model.config.vocab_size
def _green(prev_id):
    g = torch.Generator(); g.manual_seed((int(prev_id)*WM_KEY) % (2**63))
    perm = torch.randperm(V, generator=g)
    return perm[:int(GAMMA*V)]
class KGW(LogitsProcessor):
    def __call__(self, input_ids, scores):
        for b in range(input_ids.shape[0]):
            gi = _green(input_ids[b,-1].item()).to(scores.device)
            scores[b, gi] += DELTA
        return scores
def _gen(prompt, watermark):
    ids = tok(prompt, return_tensors='pt').to(device)
    lp = LogitsProcessorList([KGW()]) if watermark else None
    with torch.no_grad():
        out = model.generate(**ids, do_sample=True, top_k=50, max_new_tokens=60,
                             logits_processor=lp, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)
def kgw_z(text):
    t = tok(text, return_tensors='pt').input_ids[0].tolist()
    if len(t) < 2: return 0.0
    green = sum(1 for i in range(1,len(t)) if t[i] in set(_green(t[i-1]).tolist()))
    T = len(t)-1
    return (green - GAMMA*T)/math.sqrt(T*GAMMA*(1-GAMMA)) if T>0 else 0.0
gen = {}   # did -> {'wm':text, 'null':text}
for did, pr in prompts.items():
    gen[did] = {'wm': _gen(pr, True), 'null': _gen(pr, False)}
print('generated', len(gen), 'pairs; sample z(wm)=%.2f z(null)=%.2f' % (kgw_z(gen[list(gen)[0]]['wm']), kgw_z(gen[list(gen)[0]]['null'])))


## Cell 4 — translate all generations with real NLLB (6 conditions)
EN round-trip + EN→KO/HI/ZH/AR/DE, the same conditions as the Truthprint run.


In [ ]:
from transformers import AutoModelForSeq2SeqLM
NLLB = 'facebook/nllb-200-distilled-600M'
ntok = AutoTokenizer.from_pretrained(NLLB)
nmodel = AutoModelForSeq2SeqLM.from_pretrained(NLLB).to(device).eval()
LANG = {'ko':'kor_Hang','hi':'hin_Deva','zh':'zho_Hans','ar':'arb_Arab','de':'deu_Latn','en':'eng_Latn'}
def translate(text, src, tgt):
    ntok.src_lang = LANG[src]
    enc = ntok(text, return_tensors='pt', truncation=True, max_length=200).to(device)
    bos = ntok.convert_tokens_to_ids(LANG[tgt])
    with torch.no_grad():
        out = nmodel.generate(**enc, forced_bos_token_id=bos, max_length=220)
    return ntok.batch_decode(out, skip_special_tokens=True)[0]
conds = {}   # (did, cond) -> {'wm':..,'null':..}
for did in gen:
    conds[(did,'clean')] = gen[did]
    # round-trip EN->KO->EN
    conds[(did,'rt')] = {k: translate(translate(v,'en','ko'),'ko','en') for k,v in gen[did].items()}
    for c in ['ko','hi','zh','ar','de']:
        conds[(did,c)] = {k: translate(v,'en',c) for k,v in gen[did].items()}
print('translated conditions:', len(conds))


## Cell 5 — detect + write `05_baseline_outputs.jsonl`
Schema: one record per (method, doc, condition, watermarked?). See
`handoff/W4_SCHEMA_KO.md`. Optional MarkLLM block adds SynthID/SIR if installed.


In [ ]:
out_path = os.path.join(WORK, '05_baseline_outputs.jsonl')
recs = []
IMPL_KGW = f'self-contained KGW (Kirchenbauer 2023), gamma={GAMMA} delta={DELTA}, LLM={MODEL_ID}, NLLB=distilled-600M'
for (did, cond), texts in conds.items():
    recs.append({'method':'KGW','impl':IMPL_KGW,'doc_id':did,'condition':cond,'watermarked':True,'score':kgw_z(texts['wm'])})
    recs.append({'method':'KGW','impl':IMPL_KGW,'doc_id':did,'condition':cond,'watermarked':False,'score':kgw_z(texts['null'])})
# --- OPTIONAL: MarkLLM (SynthID / SIR). Adjust to your toolkit version. ---
try:
    subprocess.run([sys.executable,'-m','pip','install','-q','markllm'], check=False)
    from markllm.watermark.auto_watermark import AutoWatermark
    from markllm.utils.transformers_config import TransformersConfig
    tc = TransformersConfig(model=model, tokenizer=tok, vocab_size=V, device=device, max_new_tokens=60)
    for algo in ['SynthID','SIR']:
        wm = AutoWatermark.load(algo, transformers_config=tc)
        for did, pr in prompts.items():
            wtext = wm.generate_watermarked_text(pr); ntext = wm.generate_unwatermarked_text(pr)
            # translate + detect on each condition
            for cond, fn in [('clean', lambda x:x)] + [(c, (lambda cc: (lambda x: translate(x,'en',cc)))(c)) for c in ['ko','hi','zh','ar','de']] + [('rt', lambda x: translate(translate(x,'en','ko'),'ko','en'))]:
                for wmflag, txt in [(True, wtext),(False, ntext)]:
                    t = fn(txt)
                    sc = wm.detect_watermark(t); score = sc.get('score', sc.get('z_score', 0.0)) if isinstance(sc,dict) else float(sc)
                    recs.append({'method':algo,'impl':f'MarkLLM {algo}, LLM={MODEL_ID}','doc_id':did,'condition':cond,'watermarked':wmflag,'score':float(score)})
        print('MarkLLM', algo, 'done')
except Exception as e:
    print('[skip MarkLLM] ', type(e).__name__, str(e)[:160])
with open(out_path,'w',encoding='utf-8') as f:
    for r in recs: f.write(json.dumps(r, ensure_ascii=False)+'\n')
print('wrote', out_path, len(recs), 'records; methods:', sorted({r['method'] for r in recs}))


## Cell 6 — Truthprint provenance on the same source + score comparison + zip
Runs the meaning-digest provenance eval on the same facts, then the real
head-to-head scorer, then zips everything for return.


In [ ]:
# translate the source facts for Truthprint provenance (02_transformations.jsonl)
tf_path = os.path.join(WORK,'02_transformations.jsonl')
with open(tf_path,'w',encoding='utf-8') as f:
    for did, facts in facts_all:
        for i, fct in enumerate(facts):
            sid = f'{did}-s{i+1}'; en = sentence(fct)
            for c in ['ko','hi','zh','ar','de']:
                f.write(json.dumps({'transform_id':f'{sid}-{c}','sent_id':sid,'doc_id':did,'output_text':translate(en,'en',c)}, ensure_ascii=False)+'\n')
            f.write(json.dumps({'transform_id':f'{sid}-rt','sent_id':sid,'doc_id':did,'output_text':translate(translate(en,'en','ko'),'ko','en')}, ensure_ascii=False)+'\n')
import subprocess as sp
sp.run([sys.executable, os.path.join(REPO,'code','scripts','eval_provenance.py'), WORK, '--out', os.path.join(WORK,'provenance.json')], check=False)
prov = os.path.join(WORK,'provenance.json')
prov_arg = prov if os.path.exists(prov) else os.path.join(REPO,'paper','results','provenance_realmt.json')
sp.run([sys.executable, os.path.join(REPO,'code','scripts','eval_baselines_real.py'), out_path, '--truthprint', prov_arg, '--out', WORK], check=False)
import shutil
zp = shutil.make_archive(os.path.join(BASE,'w4_results'),'zip', WORK)
print('created:', zp)
print(open(os.path.join(WORK,'baselines_real.md'),encoding='utf-8').read() if os.path.exists(os.path.join(WORK,'baselines_real.md')) else 'scorer output missing')
